In [ ]:
from pathlib import Path
import json
import random
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'backend').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'backend').is_dir():
    raise FileNotFoundError('Could not find the HireAI project root. Run this notebook from within the project directory.')
DATASET_RAW = PROJECT_ROOT / "ai" / "datasets" / "raw"
DATASET_PROCESSED = PROJECT_ROOT / "ai" / "datasets" / "processed"

DATASET_RAW.mkdir(parents=True, exist_ok=True)
DATASET_PROCESSED.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

random.seed(RANDOM_SEED)

print("Project root:", PROJECT_ROOT)
print("Processed:", DATASET_PROCESSED)

In [ ]:
HIREAI_TARGET_SCHEMA = {
    "candidate": {
        "name": None,
        "email": None,
        "phone": None
    },
    "summary": None,
    "skills": [],
    "job_titles": [],
    "experience": [],
    "education": [],
    "projects": [],
    "certifications": []
}

print(
    json.dumps(
        HIREAI_TARGET_SCHEMA,
        indent=2
    )
)

In [ ]:
EXPERIENCE_SCHEMA = {
    "company": None,
    "job_title": None,
    "start_date": None,
    "end_date": None,
    "description": None
}

EDUCATION_SCHEMA = {
    "degree": None,
    "institution": None,
    "year": None
}

PROJECT_SCHEMA = {
    "name": None,
    "description": None
}

print("Experience:")
print(json.dumps(EXPERIENCE_SCHEMA, indent=2))

print("\nEducation:")
print(json.dumps(EDUCATION_SCHEMA, indent=2))

print("\nProject:")
print(json.dumps(PROJECT_SCHEMA, indent=2))

In [ ]:
def create_empty_output():
    return {
        "candidate": {
            "name": None,
            "email": None,
            "phone": None
        },
        "summary": None,
        "skills": [],
        "job_titles": [],
        "experience": [],
        "education": [],
        "projects": [],
        "certifications": []
    }

In [ ]:
def validate_training_output(output):

    required_fields = {
        "candidate",
        "summary",
        "skills",
        "job_titles",
        "experience",
        "education",
        "projects",
        "certifications"
    }

    if set(output.keys()) != required_fields:
        return False

    candidate = output["candidate"]

    if set(candidate.keys()) != {
        "name",
        "email",
        "phone"
    }:
        return False

    if not isinstance(output["skills"], list):
        return False

    if not isinstance(output["job_titles"], list):
        return False

    if not isinstance(output["experience"], list):
        return False

    if not isinstance(output["education"], list):
        return False

    if not isinstance(output["projects"], list):
        return False

    if not isinstance(output["certifications"], list):
        return False

    for exp in output["experience"]:

        if set(exp.keys()) != {
            "company",
            "job_title",
            "start_date",
            "end_date",
            "description"
        }:
            return False

    for edu in output["education"]:

        if set(edu.keys()) != {
            "degree",
            "institution",
            "year"
        }:
            return False

    for project in output["projects"]:

        if set(project.keys()) != {
            "name",
            "description"
        }:
            return False

    return True

In [ ]:
example_output = {
    "candidate": {
        "name": "Nimal Brown",
        "email": "nimal.brown@example.com",
        "phone": "+94 77 852 6747"
    },

    "summary": (
        "Python Developer with experience specializing in "
        "PostgreSQL databases, Flask-based applications, and FastAPI."
    ),

    "skills": [
        "PostgreSQL",
        "Flask",
        "FastAPI",
        "Python",
        "Git",
        "Django",
        "REST APIs"
    ],

    "job_titles": [
        "Python Developer"
    ],

    "experience": [
        {
            "company": "DataTech",
            "job_title": "Python Developer",
            "start_date": "2023-01",
            "end_date": None,
            "description": (
                "Implemented reusable software components and "
                "services. Worked with development teams to "
                "deliver new features."
            )
        },
        {
            "company": "CloudWorks",
            "job_title": "Python Developer",
            "start_date": "2022-01",
            "end_date": "2023-01",
            "description": (
                "Worked with development teams to deliver new "
                "features. Implemented REST APIs and backend "
                "services using Python, PostgreSQL, Flask, and "
                "REST APIs."
            )
        },
        {
            "company": "Smart Solutions",
            "job_title": "Python Developer",
            "start_date": "2021-01",
            "end_date": "2022-01",
            "description": (
                "Improved application performance and reliability. "
                "Designed and maintained database components using "
                "Django, PostgreSQL, FastAPI, and Python."
            )
        }
    ],

    "education": [
        {
            "degree": "Bachelor of Information Technology",
            "institution": "University of Colombo",
            "year": 2023
        }
    ],

    "projects": [
        {
            "name": "Financial Tracking Application",
            "description": (
                "Built backend services and automation solutions "
                "using FastAPI, REST APIs, and Flask."
            )
        },
        {
            "name": "Employee Management System",
            "description": (
                "Implemented application features and APIs "
                "using Git, Django, and FastAPI."
            )
        }
    ],

    "certifications": []
}

print(
    "Valid:",
    validate_training_output(example_output)
)

In [ ]:
TRAINING_INSTRUCTION = """
Extract structured information from the resume for the HireAI recruitment system.

Return ONLY valid JSON using exactly this structure:

{
  "candidate": {
    "name": null,
    "email": null,
    "phone": null
  },
  "summary": null,
  "skills": [],
  "job_titles": [],
  "experience": [],
  "education": [],
  "projects": [],
  "certifications": []
}

Rules:

1. Extract only information supported by the resume.
2. Do not invent missing information.
3. Use null for missing scalar values.
4. Use [] for missing list values.
5. job_titles must contain only the actual job title, not the company name.
6. Each experience item must contain company, job_title, start_date, end_date, and description.
7. Each education item must contain degree, institution, and year.
8. Each project must contain name and description.
9. Preserve the meaning of the original resume.
10. Return JSON only.
""".strip()

print(TRAINING_INSTRUCTION)

In [ ]:
example_resume = """
Nimal Brown
nimal.brown@example.com • +94 77 852 6747

WORK HISTORY

Python Developer at DataTech
2023-01 to Now
Implemented reusable software components and services.
Worked with development teams to deliver new features.

Python Developer at CloudWorks
01/2022 - 01/2023
Worked with development teams to deliver new features.
Implemented REST APIs and backend services using Python,
PostgreSQL, Flask, REST APIs.

Python Developer at Smart Solutions
2021-01 – 2022-01
Improved application performance and reliability.
Designed and maintained database components using Django,
PostgreSQL, FastAPI, Python.

TECHNOLOGIES

PostgreSQL • Flask • FastAPI • Python • Git • Django • REST APIs

EDUCATION

Bachelor of Information Technology - University of Colombo 2023

PROJECTS

Financial Tracking Application
Built backend services and automation solutions using FastAPI,
REST APIs, Flask.

Employee Management System
Implemented application features and APIs using Git, Django,
FastAPI.
""".strip()

In [ ]:
training_example = {
    "instruction": TRAINING_INSTRUCTION,
    "input": example_resume,
    "output": example_output
}

print(
    json.dumps(
        training_example,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
ROLE_PROFILES = {
    "Software Engineer": [
        "Python", "Java", "C++", "Git",
        "Docker", "REST APIs", "PostgreSQL", "MySQL"
    ],

    "Backend Developer": [
        "Python", "FastAPI", "Django", "Flask",
        "PostgreSQL", "MySQL", "REST APIs", "Docker"
    ],

    "Full Stack Developer": [
        "JavaScript", "TypeScript", "React", "Next.js",
        "Node.js", "PostgreSQL", "MongoDB", "Git"
    ],

    "Frontend Developer": [
        "JavaScript", "TypeScript", "React", "Next.js",
        "HTML", "CSS", "Git"
    ],

    "Python Developer": [
        "Python", "Django", "Flask", "FastAPI",
        "PostgreSQL", "REST APIs", "Git", "Docker"
    ],

    "Machine Learning Engineer": [
        "Python", "PyTorch", "TensorFlow",
        "Machine Learning", "Pandas", "NumPy",
        "Scikit-learn", "SQL"
    ],

    "Software Developer": [
        "Python", "Java", "JavaScript",
        "Git", "SQL", "REST APIs", "Docker"
    ],

    "Web Developer": [
        "HTML", "CSS", "JavaScript",
        "React", "Node.js", "Git", "SQL"
    ]
}

JOB_TITLES = list(ROLE_PROFILES.keys())

COMPANIES = [
    "Tech Solutions",
    "ABC Technologies",
    "Innovate Labs",
    "Digital Systems",
    "NextGen Software",
    "CloudWorks",
    "Smart Solutions",
    "DataTech",
    "CodeCraft Labs",
    "FutureSoft",
    "Vertex Technologies",
    "BlueWave Systems"
]

DEGREES = [
    "BSc in Computer Science",
    "BSc in Software Engineering",
    "BSc in Information Technology",
    "BSc in Computer Engineering",
    "Bachelor of Information Technology",
    "BSc in Computing and Information Systems"
]

UNIVERSITIES = [
    "University of Colombo",
    "University of Moratuwa",
    "University of Sri Jayewardenepura",
    "SLIIT",
    "NSBM Green University",
    "Sabaragamuwa University of Sri Lanka"
]

CERTIFICATIONS = [
    "AWS Certified Cloud Practitioner",
    "Microsoft Azure Fundamentals",
    "Google Associate Cloud Engineer",
    "Certified ScrumMaster",
    "TensorFlow Developer Certificate"
]

In [ ]:
FIRST_NAMES = [
    "Nimal", "Kasun", "Tharindu", "Isuru", "Dilan",
    "Sanduni", "Sachini", "Dilshan", "Kavindu",
    "John", "David", "Michael", "James", "Daniel",
    "Sarah", "Emma", "Sophia", "Olivia", "Emily",
    "Alex", "Ryan", "Kevin", "Jessica", "Laura"
]

LAST_NAMES = [
    "Perera", "Fernando", "Silva", "Jayasinghe",
    "Wijesinghe", "Bandara", "Smith", "Brown",
    "Johnson", "Williams", "Taylor", "Anderson",
    "Martin", "Thomas", "Jackson"
]

In [ ]:
def generate_candidate():
    first = random.choice(FIRST_NAMES)
    last = random.choice(LAST_NAMES)

    name = f"{first} {last}"

    email_name = (
        f"{first}.{last}"
        .lower()
        .replace(" ", "")
    )

    email = f"{email_name}@example.com"

    phone = (
        f"+94 77 "
        f"{random.randint(100,999)} "
        f"{random.randint(1000,9999)}"
    )

    return {
        "name": name,
        "email": email,
        "phone": phone
    }

In [ ]:
DESCRIPTION_TEMPLATES = [
    "Developed backend services using {skills}.",
    
    "Implemented new application features using {skills}.",
    
    "Designed and maintained software components using {skills}.",
    
    "Built REST APIs and backend services using {skills}.",
    
    "Improved application performance and reliability using {skills}.",
    
    "Worked with development teams to deliver features using {skills}.",
    
    "Designed database components and application services using {skills}.",
    
    "Participated in debugging, testing, and production support using {skills}.",
    
    "Developed scalable application components using {skills}.",
    
    "Maintained existing applications and implemented new functionality using {skills}."
]


def generate_description(skills):
    selected = random.sample(
        skills,
        min(random.randint(2, 4), len(skills))
    )

    skill_text = ", ".join(selected[:-1])

    if len(selected) > 1:
        skill_text += f", and {selected[-1]}"
    else:
        skill_text = selected[0]

    template = random.choice(
        DESCRIPTION_TEMPLATES
    )

    return template.format(
        skills=skill_text
    )

In [ ]:
def generate_experience(role, number_of_jobs):

    role_skills = ROLE_PROFILES[role]

    experiences = []

    current_year = 2026

    for i in range(number_of_jobs):

        duration = random.randint(1, 3)

        start_year = current_year - duration

        company = random.choice(COMPANIES)

        experience = {
            "company": company,
            "job_title": role,
            "start_date": f"{start_year}-01",
            "end_date": (
                None
                if i == 0
                else f"{current_year}-01"
            ),
            "description": generate_description(
                role_skills
            )
        }

        experiences.append(experience)

        current_year = start_year

    return experiences

In [ ]:
def generate_education():

    return [{
        "degree": random.choice(DEGREES),
        "institution": random.choice(UNIVERSITIES),
        "year": random.randint(2018, 2025)
    }]

In [ ]:
PROJECT_NAMES = [
    "Financial Tracking Application",
    "Employee Management System",
    "Inventory Management Platform",
    "Online Learning Platform",
    "Healthcare Management System",
    "Task Management Application",
    "E-Commerce Platform",
    "Customer Support Portal",
    "Expense Management System",
    "Job Application Platform"
]


PROJECT_TEMPLATES = [
    "Built {type} using {skills}.",
    "Developed {type} with {skills}.",
    "Implemented application features for {type} using {skills}.",
    "Created backend services for {type} using {skills}."
]


def generate_projects(role):

    role_skills = ROLE_PROFILES[role]

    projects = []

    for _ in range(random.randint(1, 2)):

        project_name = random.choice(
            PROJECT_NAMES
        )

        selected = random.sample(
            role_skills,
            min(3, len(role_skills))
        )

        description = random.choice(
            PROJECT_TEMPLATES
        ).format(
            type=project_name.lower(),
            skills=", ".join(selected)
        )

        projects.append({
            "name": project_name,
            "description": description
        })

    return projects

In [ ]:
def generate_candidate_v3():

    role = random.choice(JOB_TITLES)

    candidate = generate_candidate()

    skills = ROLE_PROFILES[role].copy()

    random.shuffle(skills)

    number_of_jobs = random.randint(1, 3)

    experience = generate_experience(
        role,
        number_of_jobs
    )

    education = generate_education()

    projects = generate_projects(role)

    certifications = []

    if random.random() < 0.35:
        certifications = [
            random.choice(CERTIFICATIONS)
        ]

    summary = (
        f"{role} with experience in "
        f"{', '.join(skills[:3])}."
    )

    return {
        "candidate": candidate,
        "summary": summary,
        "skills": skills,
        "job_titles": [role],
        "experience": experience,
        "education": education,
        "projects": projects,
        "certifications": certifications
    }

In [ ]:
def layout_standard(data):

    c = data["candidate"]

    text = (
        f"{c['name']}\n"
        f"{c['email']} | {c['phone']}\n\n"
    )

    text += "PROFESSIONAL SUMMARY\n"
    text += data["summary"] + "\n\n"

    text += "SKILLS\n"
    text += " • ".join(data["skills"]) + "\n\n"

    text += "WORK EXPERIENCE\n"

    for exp in data["experience"]:

        end = exp["end_date"] or "Present"

        text += (
            f"{exp['job_title']}\n"
            f"{exp['company']}\n"
            f"{exp['start_date']} - {end}\n"
            f"{exp['description']}\n\n"
        )

    text += "EDUCATION\n"

    for edu in data["education"]:

        text += (
            f"{edu['degree']} - "
            f"{edu['institution']} "
            f"({edu['year']})\n"
        )

    text += "\nPROJECTS\n"

    for project in data["projects"]:

        text += (
            f"{project['name']}\n"
            f"{project['description']}\n"
        )

    if data["certifications"]:

        text += "\nCERTIFICATIONS\n"

        for cert in data["certifications"]:
            text += f"- {cert}\n"

    return text.strip()

In [ ]:
def layout_compact(data):

    c = data["candidate"]

    text = (
        f"{c['name']} | "
        f"{c['email']} | "
        f"{c['phone']}\n\n"
    )

    text += "PROFILE\n"
    text += data["summary"] + "\n\n"

    text += (
        "CORE SKILLS: "
        + ", ".join(data["skills"])
        + "\n\n"
    )

    text += "EXPERIENCE\n"

    for exp in data["experience"]:

        end = exp["end_date"] or "Present"

        text += (
            f"{exp['job_title']} @ "
            f"{exp['company']} "
            f"({exp['start_date']} - {end})\n"
            f"{exp['description']}\n"
        )

    text += "\nEDUCATION\n"

    for edu in data["education"]:
        text += (
            f"{edu['degree']}, "
            f"{edu['institution']}, "
            f"{edu['year']}\n"
        )

    return text.strip()

In [ ]:
def layout_contextual(data):

    c = data["candidate"]

    text = (
        f"{c['name']}\n"
        f"{c['phone']}\n"
        f"{c['email']}\n\n"
    )

    text += "CAREER HISTORY\n\n"

    for exp in data["experience"]:

        end = exp["end_date"] or "Present"

        text += (
            f"{exp['company']}\n"
            f"{exp['job_title']}\n"
            f"{exp['start_date']} to {end}\n"
            f"{exp['description']}\n\n"
        )

    text += "EDUCATIONAL BACKGROUND\n"

    for edu in data["education"]:
        text += (
            f"{edu['institution']} - "
            f"{edu['degree']} "
            f"{edu['year']}\n"
        )

    text += "\nSELECTED PROJECTS\n"

    for project in data["projects"]:
        text += (
            f"{project['name']}: "
            f"{project['description']}\n"
        )

    return text.strip()

In [ ]:
def layout_unusual(data):

    c = data["candidate"]

    sections = []

    education_text = "EDUCATION\n"

    for edu in data["education"]:
        education_text += (
            f"{edu['degree']} | "
            f"{edu['institution']} | "
            f"{edu['year']}\n"
        )

    sections.append(education_text)

    project_text = "PROJECTS\n"

    for project in data["projects"]:
        project_text += (
            f"{project['name']} - "
            f"{project['description']}\n"
        )

    sections.append(project_text)

    experience_text = "EMPLOYMENT\n"

    for exp in data["experience"]:

        end = exp["end_date"] or "Current"

        experience_text += (
            f"{exp['company']} | "
            f"{exp['job_title']} | "
            f"{exp['start_date']} - {end}\n"
            f"{exp['description']}\n"
        )

    sections.append(experience_text)

    skills_text = (
        "TECHNICAL EXPERTISE\n"
        + " / ".join(data["skills"])
    )

    sections.append(skills_text)

    header = (
        f"{c['name']}\n"
        f"{c['email']} • {c['phone']}\n\n"
        f"{data['summary']}\n\n"
    )

    random.shuffle(sections)

    return header + "\n\n".join(sections)

In [ ]:
LAYOUTS = [
    layout_standard,
    layout_compact,
    layout_contextual,
    layout_unusual
]


def generate_resume_text(data):

    layout = random.choice(LAYOUTS)

    return layout(data)

In [ ]:
def generate_training_example():

    output = generate_candidate_v3()

    resume_text = generate_resume_text(
        output
    )

    return {
        "instruction": TRAINING_INSTRUCTION,
        "input": resume_text,
        "output": output
    }

In [ ]:
example = generate_training_example()

print(example["input"])

print("\n" + "=" * 80)
print("TARGET OUTPUT")
print("=" * 80)

print(
    json.dumps(
        example["output"],
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
print(
    "Valid:",
    validate_training_output(
        example["output"]
    )
)

In [ ]:
NUM_EXAMPLES = 1500

dataset_v3 = []

for i in range(NUM_EXAMPLES):

    example = generate_training_example()

    if validate_training_output(
        example["output"]
    ):
        dataset_v3.append(example)

    if (i + 1) % 100 == 0:
        print(
            f"Generated {i + 1}/{NUM_EXAMPLES}"
        )

print(
    "\nFinal examples:",
    len(dataset_v3)
)

In [ ]:
unique_examples = []
seen_inputs = set()

for example in dataset_v3:

    normalized_input = (
        example["input"]
        .strip()
        .lower()
    )

    if normalized_input not in seen_inputs:

        seen_inputs.add(
            normalized_input
        )

        unique_examples.append(
            example
        )

dataset_v3 = unique_examples

print(
    "Unique examples:",
    len(dataset_v3)
)

In [ ]:
V3_FILE = (
    DATASET_PROCESSED
    / "hireai_dataset_v3.jsonl"
)

with open(
    V3_FILE,
    "w",
    encoding="utf-8"
) as f:

    for example in dataset_v3:

        f.write(
            json.dumps(
                example,
                ensure_ascii=False
            )
            + "\n"
        )

print("Saved:")
print(V3_FILE)

In [ ]:
random.shuffle(dataset_v3)

n = len(dataset_v3)

train_end = int(n * 0.80)
val_end = int(n * 0.90)

train_data = dataset_v3[:train_end]
validation_data = dataset_v3[train_end:val_end]
test_data = dataset_v3[val_end:]

print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

In [ ]:
TRAIN_FILE = (
    DATASET_PROCESSED
    / "hireai_train_v3.jsonl"
)

VALIDATION_FILE = (
    DATASET_PROCESSED
    / "hireai_validation_v3.jsonl"
)

TEST_FILE = (
    DATASET_PROCESSED
    / "hireai_test_v3.jsonl"
)


def save_jsonl(data, path):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        for item in data:

            f.write(
                json.dumps(
                    item,
                    ensure_ascii=False
                )
                + "\n"
            )


save_jsonl(train_data, TRAIN_FILE)
save_jsonl(validation_data, VALIDATION_FILE)
save_jsonl(test_data, TEST_FILE)

print("Saved:")
print(TRAIN_FILE)
print(VALIDATION_FILE)
print(TEST_FILE)

In [ ]:
layout_counts = {
    "standard": 0,
    "compact": 0,
    "contextual": 0,
    "unusual": 0
}

In [ ]:
for i in range(5):

    print("=" * 80)
    print(f"EXAMPLE {i + 1}")
    print("=" * 80)

    print(
        train_data[i]["input"]
    )

    print("\nTARGET:")
    print(
        json.dumps(
            train_data[i]["output"],
            indent=2,
            ensure_ascii=False
        )
    )